In [0]:
# ------------------------------------------------------------
# SILVER 1 — Input Parameters
# ------------------------------------------------------------

dbutils.widgets.text(
    "pipeline_run_id",
    ""
)

pipeline_run_id = dbutils.widgets.get(
    "pipeline_run_id"
)

print(
    f"ADF Pipeline Run ID: {pipeline_run_id}"
)

if not pipeline_run_id:
    raise Exception(
        "SILVER_INPUT: pipeline_run_id parameter is required."
    )

In [0]:
# ------------------------------------------------------------
# SILVER 1 — Read Bronze
# ------------------------------------------------------------

from pyspark.sql.functions import col

bronze_path = (
    "abfss://bronze@olistdev1.dfs.core.windows.net/"
    "SAP/Customer"
)

df_bronze = (
    spark.read
    .format("delta")
    .load(bronze_path)
)

print(f"Bronze record count: {df_bronze.count()}")

df_bronze.printSchema()

In [0]:
# ------------------------------------------------------------
# SILVER 2 — Identify Latest Bronze Batch
# ------------------------------------------------------------

# latest_batch = (
#     df_bronze
#     .select(
#         "file_hash",
#         "pipeline_run_id",
#         "ingestion_timestamp"
#     )
#     .orderBy(
#         col("ingestion_timestamp").desc()
#     )
#     .first()
# )

# latest_file_hash = latest_batch["file_hash"]
# latest_pipeline_run_id = latest_batch["pipeline_run_id"]
# latest_ingestion_timestamp = latest_batch["ingestion_timestamp"]

# print(f"Latest file hash: {latest_file_hash}")
# print(f"Pipeline run ID: {latest_pipeline_run_id}")
# print(f"Ingestion timestamp: {latest_ingestion_timestamp}")

# ------------------------------------------------------------
# SILVER 2 — Identify Exact Bronze Batch
# ------------------------------------------------------------

# df_bronze_batch = (
#     df_bronze
#     .filter(
#         col("pipeline_run_id") == pipeline_run_id
#     )
# )

# bronze_batch_count = (
#     df_bronze_batch.count()
# )

# print(
#     f"Bronze batch record count: "
#     f"{bronze_batch_count}"
# )

# if bronze_batch_count == 0:
#     raise Exception(
#         "SILVER_BATCH: FAIL - "
#         "No Bronze records found for the supplied "
#         "pipeline_run_id."
#     )

# print(
#     "SILVER_BATCH: SUCCESS"
# )

# display(
#     df_bronze_batch.limit(5)
# )

# ------------------------------------------------------------
# Identify latest Bronze batch
# ------------------------------------------------------------

latest_batch = (
    df_bronze
    .select(
        "file_hash",
        "pipeline_run_id",
        "ingestion_timestamp"
    )
    .orderBy(
        col("ingestion_timestamp").desc()
    )
    .first()
)

if latest_batch is None:
    raise Exception(
        "SILVER_BATCH: FAIL - No Bronze data available."
    )

latest_file_hash = latest_batch["file_hash"]
latest_bronze_pipeline_run_id = latest_batch["pipeline_run_id"]

print(f"Latest Bronze file hash: {latest_file_hash}")
print(
    f"Latest Bronze pipeline_run_id: "
    f"{latest_bronze_pipeline_run_id}"
)

# ------------------------------------------------------------
# Select Bronze records for that file
# ------------------------------------------------------------

df_bronze_batch = (
    df_bronze
    .filter(
        col("file_hash") == latest_file_hash
    )
)

bronze_batch_count = df_bronze_batch.count()

print(
    f"Bronze batch record count: "
    f"{bronze_batch_count}"
)

if bronze_batch_count == 0:
    raise Exception(
        "SILVER_BATCH: FAIL - "
        "No Bronze records found for the latest file hash."
    )

Then isolate that batch:

In [0]:
df_bronze_batch = (
    df_bronze
    .filter(
        col("file_hash") == latest_file_hash
    )
)

print(
    f"Bronze batch record count: "
    f"{df_bronze_batch.count()}"
)

display(df_bronze_batch.limit(5))

In [0]:
# ------------------------------------------------------------
# SILVER 3 — Transform Bronze Batch
# ------------------------------------------------------------

from pyspark.sql.functions import (
    col,
    trim,
    upper,
    initcap,
    lpad
)

df_silver = (
    df_bronze_batch
    .withColumn(
        "customer_id",
        trim(col("customer_id"))
    )
    .withColumn(
        "customer_unique_id",
        trim(col("customer_unique_id"))
    )
    .withColumn(
        "customer_zip_code_prefix",
        lpad(
            col("customer_zip_code_prefix").cast("string"),
            5,
            "0"
        )
    )
    .withColumn(
        "customer_city",
        initcap(
            trim(col("customer_city"))
        )
    )
    .withColumn(
        "customer_state",
        upper(
            trim(col("customer_state"))
        )
    )
)

print("Silver transformation completed.")

df_silver.printSchema()

display(df_silver.limit(10))

We want:

Silver records          = 99,441
Missing columns         = 0
Null / blank values     = 0
Duplicate customer IDs  = 0
Invalid ZIP codes       = 0
### Invalid state codes     = 0

In [0]:
# ------------------------------------------------------------
# SILVER 4 — Validate Transformed Data
# ------------------------------------------------------------

from pyspark.sql.functions import (
    col,
    trim
)

print("Starting Silver validation...")

# ------------------------------------------------------------
# 1. Record count
# ------------------------------------------------------------

silver_count = df_silver.count()

print(f"Silver record count: {silver_count}")

if silver_count == 0:
    raise Exception(
        "SILVER_RECORD_COUNT: FAIL - Silver DataFrame is empty."
    )

# ------------------------------------------------------------
# 2. Required columns
# ------------------------------------------------------------

required_columns = [
    "customer_id",
    "customer_unique_id",
    "customer_zip_code_prefix",
    "customer_city",
    "customer_state"
]

missing_columns = [
    c for c in required_columns
    if c not in df_silver.columns
]

if missing_columns:
    raise Exception(
        f"SILVER_SCHEMA: FAIL - Missing columns: "
        f"{missing_columns}"
    )

print("SILVER_SCHEMA: PASS")

# ------------------------------------------------------------
# 3. Null / blank validation
# ------------------------------------------------------------

null_counts = {}

for column_name in required_columns:

    count = (
        df_silver
        .filter(
            col(column_name).isNull()
            | (trim(col(column_name).cast("string")) == "")
        )
        .count()
    )

    null_counts[column_name] = count

print(f"Null/blank validation: {null_counts}")

failed_nulls = {
    column_name: count
    for column_name, count in null_counts.items()
    if count > 0
}

if failed_nulls:
    raise Exception(
        f"SILVER_NULL_VALIDATION: FAIL - "
        f"Null or blank values found: {failed_nulls}"
    )

print("SILVER_NULL_VALIDATION: PASS")

# ------------------------------------------------------------
# 4. Duplicate customer_id validation
# ------------------------------------------------------------

duplicate_count = (
    df_silver
    .groupBy("customer_id")
    .count()
    .filter(col("count") > 1)
    .count()
)

print(f"Duplicate customer IDs: {duplicate_count}")

if duplicate_count > 0:
    raise Exception(
        f"SILVER_DUPLICATE_VALIDATION: FAIL - "
        f"{duplicate_count} duplicate customer IDs found."
    )

print("SILVER_DUPLICATE_VALIDATION: PASS")

# ------------------------------------------------------------
# 5. ZIP code validation
# ------------------------------------------------------------

invalid_zip_count = (
    df_silver
    .filter(
        ~col("customer_zip_code_prefix").rlike("^[0-9]{5}$")
    )
    .count()
)

print(f"Invalid ZIP codes: {invalid_zip_count}")

if invalid_zip_count > 0:
    raise Exception(
        f"SILVER_ZIP_VALIDATION: FAIL - "
        f"{invalid_zip_count} invalid ZIP codes found."
    )

print("SILVER_ZIP_VALIDATION: PASS")

# ------------------------------------------------------------
# 6. State validation
# ------------------------------------------------------------

valid_states = [
    "AC", "AL", "AP", "AM", "BA", "CE", "DF",
    "ES", "GO", "MA", "MT", "MS", "MG", "PA",
    "PB", "PR", "PE", "PI", "RJ", "RN", "RS",
    "RO", "RR", "SC", "SP", "SE", "TO"
]

invalid_state_count = (
    df_silver
    .filter(
        ~col("customer_state").isin(valid_states)
    )
    .count()
)

print(f"Invalid state codes: {invalid_state_count}")

if invalid_state_count > 0:
    raise Exception(
        f"SILVER_STATE_VALIDATION: FAIL - "
        f"{invalid_state_count} invalid state codes found."
    )

print("SILVER_STATE_VALIDATION: PASS")

# ------------------------------------------------------------
# Final result
# ------------------------------------------------------------

print("==========================================")
print("SILVER VALIDATION: SUCCESS")
print("==========================================")

In [0]:
# ------------------------------------------------------------
# SILVER 5 — Final Silver DataFrame - REMOVE BRONZE ONLY METADATA FIRST BEFORE PUTTIG TO SILVER
# ------------------------------------------------------------

df_silver_final = (
    df_silver
    .select(
        "customer_id",
        "customer_unique_id",
        "customer_zip_code_prefix",
        "customer_city",
        "customer_state",
        "ingestion_date"
    )
)

print("Final Silver DataFrame prepared.")

df_silver_final.printSchema()

print(
    f"Silver records ready for MERGE: "
    f"{df_silver_final.count()}"
)

display(df_silver_final.limit(10))

In [0]:
# ------------------------------------------------------------
# SILVER 6 — Check Silver Delta Target
# ------------------------------------------------------------

from delta.tables import DeltaTable

silver_path = (
    "abfss://silver@olistdev1.dfs.core.windows.net/"
    "SAP/Customer"
)

silver_exists = (
    DeltaTable.isDeltaTable(
        spark,
        silver_path
    )
)

if silver_exists:

    print(
        "SILVER TARGET: EXISTS"
    )

    df_silver_existing = (
        spark.read
        .format("delta")
        .load(silver_path)
    )

    print(
        f"Existing Silver record count: "
        f"{df_silver_existing.count()}"
    )

    df_silver_existing.printSchema()

    display(
        df_silver_existing.limit(10)
    )

else:

    print(
        "SILVER TARGET: DOES NOT EXIST"
    )

Since the Silver target does not exist, this is our first Silver load.

For the first load, we don't need a MERGE yet. We establish the Silver Delta table correctly. After that, all future batches will use MERGE.

Next step — Create Silver Delta table

In [0]:
# ------------------------------------------------------------
# SILVER 7 — Initial Load OR Incremental MERGE
# ------------------------------------------------------------

if not silver_exists:

    print(
        "Silver target does not exist."
    )

    print(
        "Performing initial Silver load..."
    )

    (
        df_silver_final
        .write
        .format("delta")
        .mode("overwrite")
        .option(
            "overwriteSchema",
            "true"
        )
        .partitionBy(
            "ingestion_date"
        )
        .save(silver_path)
    )

    print(
        "=========================================="
    )

    print(
        "SILVER INITIAL LOAD: SUCCESS"
    )

    print(
        f"Silver path: {silver_path}"
    )

    print(
        f"Records written: "
        f"{df_silver_final.count()}"
    )

    print(
        "=========================================="
    )

else:

    print(
        "Silver target already exists."
    )

    print(
        "Incremental MERGE will be executed."
    )

Now we move to the part that makes this pipeline truly incremental and production-oriented: the conditional Delta MERGE.

But before implementing it against the real Silver table, we're going to build the change-detection logic first.

Next step — Prepare the MERGE source

In [0]:
# ------------------------------------------------------------
# SILVER 8 — Prepare MERGE Source
# ------------------------------------------------------------

from pyspark.sql.functions import (
    col,
    sha2,
    concat_ws,
    coalesce,
    lit
)

# Create a hash of the business attributes.
# ingestion_date is intentionally excluded because it is
# technical metadata, not a customer change.

df_merge_source = (
    df_silver_final
    .withColumn(
        "record_hash",
        sha2(
            concat_ws(
                "||",
                coalesce(col("customer_unique_id"), lit("")),
                coalesce(col("customer_zip_code_prefix"), lit("")),
                coalesce(col("customer_city"), lit("")),
                coalesce(col("customer_state"), lit(""))
            ),
            256
        )
    )
)

print("MERGE source prepared.")

df_merge_source.printSchema()

print(
    f"MERGE source records: "
    f"{df_merge_source.count()}"
)

display(
    df_merge_source.select(
        "customer_id",
        "customer_unique_id",
        "customer_zip_code_prefix",
        "customer_city",
        "customer_state",
        "ingestion_date",
        "record_hash"
    ).limit(10)
)

In [0]:
# ------------------------------------------------------------
# SILVER 9 — Incremental Delta MERGE
# ------------------------------------------------------------

if silver_exists:

    from delta.tables import DeltaTable

    silver_table = DeltaTable.forPath(
        spark,
        silver_path
    )

    print("Starting Silver MERGE...")

    (
        silver_table.alias("target")
        .merge(
            df_merge_source.alias("source"),
            "target.customer_id = source.customer_id"
        )

        # ----------------------------------------------------
        # Existing customer — update ONLY when business
        # data changed
        # ----------------------------------------------------

        .whenMatchedUpdate(
            condition="""
                target.customer_unique_id <> source.customer_unique_id
                OR target.customer_zip_code_prefix <> source.customer_zip_code_prefix
                OR target.customer_city <> source.customer_city
                OR target.customer_state <> source.customer_state
            """,
            set={
                "customer_unique_id": "source.customer_unique_id",
                "customer_zip_code_prefix": "source.customer_zip_code_prefix",
                "customer_city": "source.customer_city",
                "customer_state": "source.customer_state",
                "ingestion_date": "source.ingestion_date"
            }
        )

        # ----------------------------------------------------
        # New customer — INSERT
        # ----------------------------------------------------

        .whenNotMatchedInsert(
            values={
                "customer_id": "source.customer_id",
                "customer_unique_id": "source.customer_unique_id",
                "customer_zip_code_prefix": "source.customer_zip_code_prefix",
                "customer_city": "source.customer_city",
                "customer_state": "source.customer_state",
                "ingestion_date": "source.ingestion_date"
            }
        )

        .execute()
    )

    print("==========================================")
    print("SILVER MERGE: SUCCESS")
    print("==========================================")

else:

    print(
        "Silver target does not exist. "
        "MERGE skipped because initial load is required."
    )

TEESTING MERGE BY COPYING SAME DATA INTO SAME TABLE WE EXPECTING NO DUPES

In [0]:
# ------------------------------------------------------------
# SILVER 10 — Verify MERGE Result
# ------------------------------------------------------------

df_silver_after_merge = (
    spark.read
    .format("delta")
    .load(silver_path)
)

silver_count = df_silver_after_merge.count()

distinct_customer_count = (
    df_silver_after_merge
    .select("customer_id")
    .distinct()
    .count()
)

duplicate_customer_count = (
    df_silver_after_merge
    .groupBy("customer_id")
    .count()
    .filter(col("count") > 1)
    .count()
)

print(f"Silver record count: {silver_count}")
print(f"Distinct customer IDs: {distinct_customer_count}")
print(f"Duplicate customer IDs: {duplicate_customer_count}")

print("\nSilver ingestion dates:")
display(
    df_silver_after_merge
    .groupBy("ingestion_date")
    .count()
    .orderBy("ingestion_date")
)

TESYINMG WITH SAMEE ID BUT UPDATED REC

BELOW IS OUTDATED


removing dupes here before starting silver

# transformations

c.1

c.2

In [0]:
df_silver.printSchema()


create finalsilver projetion

CREATING EXT LOC FOR SILVER AND GRANTING MYSELF ACCESS

In [0]:
# %sql
# GRANT READ FILES
# ON EXTERNAL LOCATION extloc_olist_silver
# TO `ayushmanpandita1999@gmail.com`;

# GRANT WRITE FILES
# ON EXTERNAL LOCATION extloc_olist_silver
# TO `ayushmanpandita1999@gmail.com`;

In [0]:
# Silver final verification

df_silver_check = (
    spark.read
    .format("delta")
    .load(silver_path)
)

total_count = df_silver_check.count()

distinct_customer_count = (
    df_silver_check
    .select("customer_id")
    .distinct()
    .count()
)

duplicate_customer_count = (
    df_silver_check
    .groupBy("customer_id")
    .count()
    .filter(col("count") > 1)
    .count()
)

print("==========================================")
print("SILVER VERIFICATION")
print("==========================================")
print(f"Total Silver records: {total_count}")
print(f"Distinct customer IDs: {distinct_customer_count}")
print(f"Duplicate customer IDs: {duplicate_customer_count}")

Write Customer Silver as Delta

Create a temporary MERGE test table

In [0]:
df_silver_check.filter(
    col("customer_city") == "test_city"
).select(
    "customer_id",
    "customer_unique_id",
    "customer_city",
    "customer_state",
    "ingestion_date"
).show(truncate=False)

In [0]:
df_silver_check.filter(
    col("customer_city") ==  "Test_city"
).select(
    "customer_id",
    "customer_unique_id",
    "customer_city",
    "customer_state",
    "ingestion_date"
).show(truncate=False)